In [1]:
import cv2
import os

In [2]:
folder_name = os.listdir('SL')
folder_name = sorted(folder_name)
print(len(folder_name))
print(folder_name)

98
['afternoon', 'angry', 'answer', 'apple', 'baby', 'bad', 'bank', 'bathroom', 'bed', 'bike', 'book', 'boy', 'bread', 'brother', 'bus', 'bye', 'car', 'chair', 'child', 'city', 'class', 'coffee', 'college', 'come', 'computer', 'day', 'doctor', 'door', 'drink', 'eat', 'evening', 'family', 'father', 'fine', 'food', 'friend', 'fruit', 'girl', 'go', 'good', 'goodbye', 'happy', 'help', 'home', 'hospital', 'house', 'internet', 'kitchen', 'learn', 'love', 'man', 'medicine', 'milk', 'money', 'morning', 'mother', 'music', 'night', 'no', 'nurse', 'office', 'paper', 'phone', 'please', 'police', 'question', 'read', 'road', 'room', 'run', 'sad', 'school', 'shop', 'sister', 'sit', 'sleep', 'smile', 'sorry', 'stand', 'stop', 'student', 'study', 'table', 'tea', 'teacher', 'time', 'today', 'tomorrow', 'train', 'wait', 'walk', 'water', 'welcome', 'woman', 'work', 'write', 'yes', 'yesterday']


In [3]:
frame_total = 25

for i in folder_name:
    video_list = os.listdir('SL/' + i)
    video_list = sorted(video_list)

    for j in video_list:
        video_path = 'SL/' + i + '/' + j
        img_name = j.replace('.mp4', '')
        imag_folder = 'frame/' + i + '/' + img_name
        os.makedirs(imag_folder, exist_ok=True)

        video = cv2.VideoCapture(video_path)
        if not video.isOpened():
            print(video_path, 'failed to open')
            continue

        total_frame = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frame == 0:
            print(video_path, 'no frames found')
            video.release()
            continue

        gap = total_frame / frame_total
        indices = []
        for i2 in range(frame_total):
            frame_no = int(i2 * gap)
            indices.append(frame_no)

        k = 0
        saved = 0
        while True:
            ret, frame = video.read()
            if ret == False or frame is None:
                break

            if k in indices:
                file_name = imag_folder + '/frame' + str(saved) + '.jpg'
                cv2.imwrite(file_name, frame)
                saved = saved + 1

            k = k + 1

        video.release()

    print(i, 'folder done')

afternoon folder done
angry folder done
answer folder done
apple folder done
baby folder done


[h264 @ 0x113ac0b10] Invalid NAL unit size (745 > 472).
[h264 @ 0x113ac0b10] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x113e11a90] stream 1, offset 0x3b468: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x113e11a90] stream 1, offset 0x3b7d3: partial file


bad folder done
bank folder done
bathroom folder done
bed folder done
bike folder done
book folder done
boy folder done
bread folder done
brother folder done
bus folder done
bye folder done
car folder done
chair folder done
child folder done
city folder done
class folder done
coffee folder done
college folder done
come folder done
computer folder done
day folder done
doctor folder done
door folder done
drink folder done
eat folder done
evening folder done
family folder done
father folder done
fine folder done
food folder done
friend folder done
fruit folder done
girl folder done
go folder done
good folder done
goodbye folder done
happy folder done
help folder done
home folder done
hospital folder done
house folder done
internet folder done
kitchen folder done
learn folder done
love folder done
man folder done
medicine folder done
milk folder done
money folder done
morning folder done
mother folder done
music folder done
night folder done
no folder done
nurse folder done
office folder d

In [13]:
import numpy as np
import mediapipe as mp
import os
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)

X_data = []
y_data = []

for folder in folder_name:
    video_folders = sorted(os.listdir(os.path.join('frame', folder)))

    for sub_folder in video_folders:
        frame_path = os.path.join('frame', folder, sub_folder)
        frame_files = sorted(os.listdir(frame_path))

        sequence = []
        for img_file in frame_files:
            fp = os.path.join(frame_path, img_file)
            if not os.path.isfile(fp) or not img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                continue

            try:
                img = mp.Image.create_from_file(fp)
                result = detector.detect(img)
            except:
                result = None

            left_hand = [0.0] * 63
            right_hand = [0.0] * 63

            if result and getattr(result, 'hand_landmarks', None):
                for index, hand in enumerate(result.hand_landmarks):
                    hand_side = result.handedness[index][0].category_name
                    wrist = hand[0]
                    temp_points = []
                    
                    for point in hand:
                        temp_points.extend([point.x - wrist.x, point.y - wrist.y, point.z - wrist.z])

                    max_value = max([abs(x) for x in temp_points])
                    if max_value > 0.0:
                        temp_points = [x / max_value for x in temp_points]

                    if hand_side == 'Left':
                        left_hand = temp_points
                    elif hand_side == 'Right':
                        right_hand = temp_points

            combined_data = left_hand + right_hand
            sequence.append(combined_data)

        while len(sequence) < 25:
            sequence.append([0.0] * 126)
        sequence = sequence[:25]

        X_data.append(sequence)
        y_data.append(folder)

    print(folder, 'done')

X_data = np.array(X_data)
y_data = np.array(y_data)

np.save('X_data.npy', X_data)
np.save('y_data.npy', y_data)

I0000 00:00:1783680751.224932  446065 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
W0000 00:00:1783680751.244642  446073 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783680751.261152  446072 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


afternoon done
angry done
answer done
apple done
baby done
bad done
bank done
bathroom done
bed done
bike done
book done
boy done
bread done
brother done
bus done
bye done
car done
chair done
child done
city done
class done
coffee done
college done
come done
computer done


E0000 00:00:1783680874.069446  446066 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-10T16:38:34.063525+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


day done
doctor done
door done
drink done
eat done
evening done
family done
father done
fine done
food done
friend done
fruit done


E0000 00:00:1783680934.078788  446066 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-10T16:38:34.063525+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


...

E0000 00:00:1783681114.094944  446066 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-10T16:38:34.063525+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


sleep done
smile done
sorry done
stand done
stop done
student done
study done
table done
tea done
teacher done
time done
today done
tomorrow done


E0000 00:00:1783681174.100009  446066 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-10T16:38:34.063525+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


train done
wait done
walk done
water done
welcome done
woman done
work done
write done
yes done
yesterday done


E0000 00:00:1783681234.105312  446066 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-10T16:38:34.063525+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


In [14]:
import numpy as np

X = np.load('X_data.npy')
y = np.load('y_data.npy')

print(f"Original Data: {X.shape[0]} videos, {len(np.unique(y))} words")

valid_indices = []
for i in range(len(X)):
    if not np.all(X[i] == 0.0):
        valid_indices.append(i)

X_clean = X[valid_indices]
y_clean = y[valid_indices]
print(f"After removing blanks: {X_clean.shape[0]} videos")

X_augmented_list = [X_clean]
y_augmented_list = [y_clean]

for i in range(5):
    noise = np.random.normal(0, 0.015, X_clean.shape)
    X_noisy = X_clean + noise
    X_augmented_list.append(X_noisy)
    y_augmented_list.append(y_clean)

X_final = np.vstack(X_augmented_list)
y_final = np.concatenate(y_augmented_list)

print(f"Final Augmented Data: {X_final.shape[0]} videos, {len(np.unique(y_final))} words")

np.save('X_train_ready.npy', X_final)
np.save('y_train_ready.npy', y_final)
print("Data is now ready for high accuracy training!")

Original Data: 888 videos, 98 words
After removing blanks: 790 videos
Final Augmented Data: 4740 videos, 98 words
Data is now ready for high accuracy training!


In [15]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import joblib

X = np.load('X_train_ready.npy')
y = np.load('y_train_ready.npy')

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
y_cat = to_categorical(y_encoded)
joblib.dump(encoder, 'label_encoder.pkl')

X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.15, random_state=42)

model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(25, 126)),
    Dropout(0.5),
    
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    
    Dense(64, activation='relu'),
    Dense(y_cat.shape[1], activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001)

print("Training LSTM Motion Model...")
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stop, reduce_lr])

loss, accuracy = model.evaluate(X_test, y_test)
print(f"\nFinal Test Accuracy: {accuracy * 100:.2f}%")
model.save('sign_language_model_100.keras')

Training LSTM Motion Model...
Epoch 1/100


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


126/126 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.0563 - loss: 4.2120 - val_accuracy: 0.1153 - val_loss: 3.6143 - learning_rate: 0.0010
Epoch 2/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.1382 - loss: 3.3640 - val_accuracy: 0.2039 - val_loss: 2.9943 - learning_rate: 0.0010
Epoch 3/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.2142 - loss: 2.8515 - val_accuracy: 0.2968 - val_loss: 2.4283 - learning_rate: 0.0010
Epoch 4/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.2852 - loss: 2.4422 - val_accuracy: 0.3868 - val_loss: 2.0837 - learning_rate: 0.0010
Epoch 5/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.3460 - loss: 2.1369 - val_accuracy: 0.5007 - val_loss: 1.7221 - learning_rate: 0.0010
Epoch 6/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.4396 - loss: 1.8188 - val_accuracy: 0.5570 - val_loss: 1.4551 - learning_rate: 0.0010
Epoch 7/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.4644 - loss: 1.6591 

E0000 00:00:1783681294.044248  446066 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-10T16:38:34.063525+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8583 - loss: 0.4309 - val_accuracy: 0.9606 - val_loss: 0.1643 - learning_rate: 0.0010
Epoch 21/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8739 - loss: 0.3841 - val_accuracy: 0.9395 - val_loss: 0.1815 - learning_rate: 0.0010
Epoch 22/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8727 - loss: 0.3994 - val_accuracy: 0.9297 - val_loss: 0.2178 - learning_rate: 0.0010
Epoch 23/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8707 - loss: 0.4151 - val_accuracy: 0.9578 - val_loss: 0.1433 - learning_rate: 0.0010
Epoch 24/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8841 - loss: 0.3665 - val_accuracy: 0.9606 - val_loss: 0.1317 - learning_rate: 0.0010
Epoch 25/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8871 - loss: 0.3567 - val_accuracy: 0.9747 - val_loss: 0.1027 - learning_rate: 0.0010
Epoch 26/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9015 - loss: 0

E0000 00:00:1783681354.053970  446066 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-10T16:38:34.063525+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9814 - loss: 0.0610 - val_accuracy: 0.9944 - val_loss: 0.0101 - learning_rate: 2.5000e-04
Epoch 53/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9821 - loss: 0.0605 - val_accuracy: 0.9944 - val_loss: 0.0109 - learning_rate: 2.5000e-04
Epoch 54/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9806 - loss: 0.0613 - val_accuracy: 0.9958 - val_loss: 0.0095 - learning_rate: 2.5000e-04
Epoch 55/100
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9816 - loss: 0.0585 - val_accuracy: 0.9944 - val_loss: 0.0098 - learning_rate: 2.5000e-04
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9972 - loss: 0.0143

Final Test Accuracy: 99.72%


E0000 00:00:1783681414.062498  446066 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-10T16:38:34.063525+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1783681474.067469  446066 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-10T16:38:34.063525+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1783681534.072946  446066 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-10T16:38:34.063525+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
